In [1]:
import os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb
import xgboost as xgb

SEED = 42
PROCESSED_DATA_PATH = "../processed-data"
ORIGINAL_DATA_PATH = "../original-data"

In [3]:
print("1. Đọc dữ liệu Preprocessed...")
df = pd.read_csv(f'{PROCESSED_DATA_PATH}/final_training_data.csv')
df['Date'] = pd.to_datetime(df['Date'])

# Ép kiểu Category cho các cột chuỗi
object_cols = [col for col in df.select_dtypes(include=['object']).columns if col not in ['Date', 'Split']]
for col in object_cols:
    df[col] = df[col].astype('category')

print("2. Chuẩn bị tập Train toàn phần (2012-2022)...")
train_full = df[df['Split'] == 'Train'].copy()
test_data = df[df['Split'] == 'Test'].copy()

cols_to_drop = ['Date', 'Split', 'Revenue', 'COGS']
features = [col for col in df.columns if col not in cols_to_drop]

X_train_full = train_full[features].copy()

# Áp dụng Log Transform cho Target
y_train_rev = np.log1p(train_full['Revenue'])
y_train_cogs = np.log1p(train_full['COGS'])

# Thêm cột predicted_revenue cho quá trình train COGS
X_train_cogs = X_train_full.copy()
X_train_cogs['predicted_revenue'] = train_full['Revenue']

# ==========================================
# 3. HUẤN LUYỆN LẠI TRÊN TOÀN BỘ DỮ LIỆU
# ==========================================
lgb_params = {
    'objective': 'regression', 'metric': 'mae', 'learning_rate': 0.02, 
    'num_leaves': 63, 'min_child_samples': 15, 'subsample': 0.8, 'colsample_bytree': 0.7,
    'reg_alpha': 0.1, 'reg_lambda': 0.5, 'n_estimators': 3000, 'random_state': SEED, 'n_jobs': -1, 'verbose': -1
}

xgb_params = {
    'objective': 'reg:squarederror', 'eval_metric': 'mae', 'learning_rate': 0.02, 
    'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 0.7, 'alpha': 0.1, 'lambda': 0.5,
    'n_estimators': 3000, 'random_state': SEED, 'n_jobs': -1, 'enable_categorical': True
}

# TRỌNG SỐ TỐI ƯU TỪ NOTEBOOK 2
W_REV_LGB = 0.50 
W_REV_XGB = 0.50

W_COGS_LGB = 0.35
W_COGS_XGB = 0.65

print("-> Huấn luyện Ensemble cho Revenue...")
model_rev_lgb = lgb.LGBMRegressor(**lgb_params).fit(X_train_full, y_train_rev)
model_rev_xgb = xgb.XGBRegressor(**xgb_params).fit(X_train_full, y_train_rev, verbose=False)

print("-> Huấn luyện Ensemble cho COGS...")
model_cogs_lgb = lgb.LGBMRegressor(**lgb_params).fit(X_train_cogs, y_train_cogs)
model_cogs_xgb = xgb.XGBRegressor(**xgb_params).fit(X_train_cogs, y_train_cogs, verbose=False)

# ==========================================
# 4. DỰ BÁO ĐỆ QUY (RECURSIVE FORECASTING)
# ==========================================
print("\n4. Bắt đầu dự báo đệ quy cho 548 ngày tập Test...")

test_data = test_data.sort_values('Date').reset_index(drop=True)
full_data_recursive = pd.concat([train_full, test_data]).sort_values('Date').reset_index(drop=True)

test_start_idx = full_data_recursive[full_data_recursive['Split'] == 'Test'].index[0]

# Giới hạn an toàn (Clipping): Không cho doanh thu vượt quá 1.5 lần đỉnh lịch sử
max_historical_rev = train_full['Revenue'].max() * 1.5
max_historical_cogs = train_full['COGS'].max() * 1.5

for i in range(test_start_idx, len(full_data_recursive)):
    current_date = full_data_recursive.loc[i, 'Date']
    
    # 4.1 CẬP NHẬT CÁC BIẾN LAGS
    for lag in [1, 7, 14, 30, 365]:
        full_data_recursive.loc[i, f'rev_lag_{lag}'] = full_data_recursive.loc[i-lag, 'Revenue']
        full_data_recursive.loc[i, f'cogs_lag_{lag}'] = full_data_recursive.loc[i-lag, 'COGS']
    
    # 4.2 CẬP NHẬT ROLLING WINDOWS (Trung bình và độ lệch chuẩn)
    # Lấy dữ liệu của 30 ngày ngay trước đó
    past_30_days_rev = full_data_recursive.loc[i-30:i-1, 'Revenue']
    past_30_days_cogs = full_data_recursive.loc[i-30:i-1, 'COGS']
    
    for window in [7, 14, 30]:
        full_data_recursive.loc[i, f'rev_rolling_mean_{window}'] = past_30_days_rev.tail(window).mean()
        full_data_recursive.loc[i, f'cogs_rolling_mean_{window}'] = past_30_days_cogs.tail(window).mean()
        
        full_data_recursive.loc[i, f'rev_rolling_std_{window}'] = past_30_days_rev.tail(window).std()
        full_data_recursive.loc[i, f'cogs_rolling_std_{window}'] = past_30_days_cogs.tail(window).std()
    
    # 4.3 DỰ BÁO REVENUE
    X_current = full_data_recursive.loc[[i], features]
    
    pred_log_rev_lgb = model_rev_lgb.predict(X_current)[0]
    pred_log_rev_xgb = model_rev_xgb.predict(X_current)[0]
    
    # Chuyển ngược Log -> Số thực và tính trọng số
    pred_rev = W_REV_LGB * np.expm1(pred_log_rev_lgb) + W_REV_XGB * np.expm1(pred_log_rev_xgb)
    
    # Áp dụng Clipping an toàn
    pred_rev = np.clip(pred_rev, 0, max_historical_rev)
    full_data_recursive.loc[i, 'Revenue'] = pred_rev
    
    # 4.4 DỰ BÁO COGS (SEQUENTIAL: Dùng Revenue vừa tính được)
    X_current_cogs = X_current.copy()
    X_current_cogs['predicted_revenue'] = pred_rev
    
    pred_log_cogs_lgb = model_cogs_lgb.predict(X_current_cogs)[0]
    pred_log_cogs_xgb = model_cogs_xgb.predict(X_current_cogs)[0]
    
    pred_cogs = W_COGS_LGB * np.expm1(pred_log_cogs_lgb) + W_COGS_XGB * np.expm1(pred_log_cogs_xgb)
    pred_cogs = np.clip(pred_cogs, 0, max_historical_cogs)
    
    full_data_recursive.loc[i, 'COGS'] = pred_cogs
    
    if i % 100 == 0:
        print(f" Đã dự báo xong đến ngày: {current_date.date()}")

# ==========================================
# 5. XUẤT FILE SUBMISSION
# ==========================================
submission_final = full_data_recursive[full_data_recursive['Split'] == 'Test'][['Date', 'Revenue', 'COGS']]
submission_final['Date'] = submission_final['Date'].dt.strftime('%Y-%m-%d')
submission_final['Revenue'] = submission_final['Revenue'].round(2)
submission_final['COGS'] = submission_final['COGS'].round(2)

submission_final.to_csv('final_submission.csv', index=False)
submission_final.head()

1. Đọc dữ liệu Preprocessed...
2. Chuẩn bị tập Train toàn phần (2012-2022)...
-> Huấn luyện Ensemble cho Revenue...
-> Huấn luyện Ensemble cho COGS...

4. Bắt đầu dự báo đệ quy cho 548 ngày tập Test...
 Đã dự báo xong đến ngày: 2023-02-02
 Đã dự báo xong đến ngày: 2023-05-13
 Đã dự báo xong đến ngày: 2023-08-21
 Đã dự báo xong đến ngày: 2023-11-29
 Đã dự báo xong đến ngày: 2024-03-08
 Đã dự báo xong đến ngày: 2024-06-16


,Date,Revenue,COGS
3468,2023-01-01,1688444.07,1343218.34
3469,2023-01-02,1362384.68,1060725.87
3470,2023-01-03,1144782.98,950424.10
3471,2023-01-04,1056457.49,876903.73
3472,2023-01-05,1144278.78,990798.55
